# General 1D CSD pipeline

Reusable version of the session notebooks (`CSD 1D 230516`, `230606`, `230608`, `250320`, …).

**Expected files in each dataset folder**
- `lfp.npy` — `(n_stim, n_chan, n_time)`
- `x_coords.npy`, `y_coords.npy`
- `T0.npy` — stimulus onset sample

**What this does**
1. Drop bad channels → re-reference → slice channel range (order fixed vs older notebooks)
2. Epoch around `T0`, baseline-correct, trial-average
3. 1D **kCSD** per shank (`unique x`) and/or **finite-difference** CSD
4. Save `.npy` outputs + example plots

Edit the **DATASETS** list below for new sessions, then run all cells.


In [ ]:
# Optional: install deps once
# %pip install numpy scipy matplotlib kcsd


In [ ]:
import os
import sys
from pathlib import Path

# Make the sibling module importable when the notebook cwd is this folder
HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from csd_1d_pipeline import (
    DatasetConfig,
    EXAMPLE_DATASETS,
    merge_config,
    run_dataset,
    run_many,
)

print("Pipeline ready.")


## 1) Shared defaults + new dataset list

Only change what differs per session. Paths can be Windows UNC or a local copy.


In [ ]:
# Shared defaults for a new experiment batch
DEFAULTS = dict(
    fs=2500.0,
    t_pre=200.0,
    t_post=800.0,
    step_ms=5.0,
    ref_chan="deepest",       # or int index, "shallowest", None
    channels_remove=(),       # e.g. (191,)
    chan_start=None,          # None = from 0
    chan_end=None,            # None = all remaining channels
    compute_kcsd=True,
    compute_fd=True,
    y_scale="mm",             # "mm" (recommended) or "unit" ([0,1])
    h=0.25,
    n_src_factor=4.0,
    smooth_sigma_time=0.0,    # set 1.0 to match older notebooks
    save=True,
    save_plots=True,
    plot=True,                # show figures in the notebook
)

# ---------------------------------------------------------------------------
# NEW DATASETS — edit / add rows here
# ---------------------------------------------------------------------------
# Each dict needs at least "folder". Everything else overrides DEFAULTS.
DATASETS = [
    # {
    #     "name": "sessionA_USV",
    #     "folder": r"//experimentfs.bccn-berlin.pri/experiment/share/Miguel/"
    #               r"LOMBARD EFFECT NPX files and analysis/CSD analysis/NEWID/USV",
    #     "chan_start": 0,
    #     "chan_end": 384,
    #     "channels_remove": (),
    # },
    # {
    #     "name": "sessionB_TONE",
    #     "folder": r"/path/to/local/copy/sessionB",
    #     "chan_start": 75,
    #     "chan_end": 275,
    #     "channels_remove": (191,),
    #     "ref_chan": 0,
    # },
]

# To re-run the sessions from the uploaded notebooks instead, uncomment:
# DATASETS = EXAMPLE_DATASETS

assert DATASETS, (
    "DATASETS is empty. Add folders above, or set DATASETS = EXAMPLE_DATASETS "
    "to use the previous Lombard sessions."
)

for d in DATASETS:
    print(d.get("name", Path(d["folder"]).name), "->", d["folder"])


## 2) Run all datasets


In [ ]:
results = run_many(DATASETS, defaults=DEFAULTS, stop_on_error=False)

ok = [r for r in results if "error" not in r]
bad = [r for r in results if "error" in r]
print(f"\nFinished: {len(ok)} ok, {len(bad)} failed")
for r in bad:
    print(" -", r["name"], ":", r["error"])


## 3) Quick inspection of the last successful run


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if not ok:
    raise RuntimeError("No successful runs to inspect.")

r = ok[-1]
print("Inspecting:", r["name"])
print("evoked", r["evoked"].shape, "median spacing", r["median_spacing_um"], "µm")

plt.figure(figsize=(10, 5))
plt.imshow(
    r["evoked"],
    origin="lower",
    aspect="auto",
    extent=[r["time_ms"][0], r["time_ms"][-1], 0, r["evoked"].shape[0]],
    cmap="RdBu_r",
)
plt.colorbar(label="LFP")
plt.xlabel("Time (ms)")
plt.ylabel("Channel (selected)")
plt.title(f'{r["name"]} — evoked LFP')
plt.show()

if r["csd_kcsd"]:
    ux = sorted(r["csd_kcsd"])[0]
    print("kCSD columns:", sorted(r["csd_kcsd"]), "example x=", ux, "shape", r["csd_kcsd"][ux].shape)


## 4) Single-folder smoke test (optional)

Run one folder interactively without editing the batch list.


In [ ]:
# SINGLE = dict(
#     name="debug",
#     folder=r"/path/to/one/dataset",
#     chan_start=0,
#     chan_end=96,
# )
# single_result = run_dataset(merge_config(DEFAULTS, SINGLE))


## Outputs written per folder

- `csd_1d/csd_x{X}.npy`, `y_x{X}.npy`, `V_x{X}.npy`, `time.npy`
- `csd_1d_array.npy`, `csd_1d_x.npy`, `csd_1d_time.npy` (MATLAB-friendly stack)
- `csd_fd_array.npy` / `csd_fd_x{X}.npy` if finite-difference is enabled
- example PNGs under `csd_1d/`

CLI alternative from a terminal:

```bash
python csd_1d_pipeline.py --folder "/path/to/ds1" --folder "/path/to/ds2" --ref deepest
```
